In [36]:
from openai import OpenAI
openai_client = OpenAI()

In [35]:
from dotenv import load_dotenv
load_dotenv()

True

In [1]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [2]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1342

In [3]:
documents[750]

{'id': 'ba50d9e0bc',
 'course': 'mlops-zoomcamp',
 'section': 'Module 3: Orchestration',
 'question': 'Mage in Codespaces in a subfolder under mlops-zoomcamp repository',
 'answer': '**Issue 1:** Errors such as:\n\n```bash\n[+] Running 1/1\n\n✘ magic-database Error too many requests: You have reached your pull rate limit. You may increase the limit by authenticating and upgra...                   \n\nError response from daemon: too many requests: You have reached your pull rate limit. You may increase the limit by authenticating and upgrading: [docker.com](https://www.docker.com/increase-rate-limit)\n```\n\n**Issue 2:** Popups with different percentage values indicating space is in single digits.\n\n<{IMAGE:image_1}>\n\n**Solution:** It is not recommended to set up Mage as a subfolder of mlops-zoomcamp. See findings in this thread for more information.'}

In [4]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [8]:
index.search("What is the refund policy?", num_results=3)

[{'id': 'ec7b94d408',
  'course': 'machine-learning-zoomcamp',
  'section': 'Module 9. Serverless Deep Learning',
  'question': 'What IAM permission policy is needed to complete Week 9: Serverless?',
  'answer': '1. **Sign in to the AWS Console**: Log in to the AWS Console.\n\n2. **Navigate to IAM**: Go to the IAM service by clicking on "Services" in the top left corner and selecting "IAM" under the "Security, Identity, & Compliance" section.\n\n3. **Create a new policy**:\n   - In the left navigation pane, select "Policies" and click on "Create policy."\n\n4. **Select the service and actions**:\n   - Click on "JSON" and copy and paste the JSON policy for the specific ECR actions.\n\n```json\n{\n  "Version": "2012-10-17",\n  "Statement": [\n    {\n      "Sid": "VisualEditor0",\n      "Effect": "Allow",\n      "Action": [\n        "ecr:CreateRepository",\n        "ecr:GetAuthorizationToken",\n        "ecr:BatchCheckLayerAvailability",\n        "ecr:BatchGetImage",\n        "ecr:Initiate

In [9]:
question = 'I just discovered the course, can I still join?'

In [11]:
index.search(
    question,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '9f689c185f',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I missed the first homework - can I still get a certificate?',
  'answer': 'Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learn

In [16]:
def search(question, course = "llm-zoomcamp"):
    filter_dict = {'course': course}
    return index.search(
        question,
        filter_dict=filter_dict,
        num_results=5
    )

In [21]:
search_results = search(question)

In [18]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

In [19]:
USER_PROMPT_TEMPLATE = '''

Question:
{question}

Context:
{context}
'''

In [20]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [28]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(question=question, context=context)
    return prompt.strip()


In [30]:
prompt = build_prompt(question, search_results)
print(prompt)

Question:
I just discovered the course, can I still join?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: I missed the first homework - can I still get a certificate?
A: Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related

In [31]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [38]:
ans = llm(prompt)
print(ans)

Yes — you can still join and start learning now.

If you want a certificate, the key requirement is that you submit your project while submissions are still open. Also, certificates are only available for the live cohort, not self-paced mode.


In [39]:
response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )

In [ ]:
response.output_text

'Yes — you can still join and start learning right away. You don’t need a confirmation email or prior registration.\n\nIf you want a certificate, make sure to submit your project while submissions are still open, and you need to complete the course with the live cohort.'

In [42]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=message_history
)

In [43]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [44]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [45]:
answer = rag(question)
print(answer)

Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.
